In [7]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"X:\dev\projects\study\ATLAS")
VENV_DIR = PROJECT_ROOT / ".venv"
MODELS_DIR = PROJECT_ROOT / "models"
LLAMA_DIR = PROJECT_ROOT / "llama"
EVAL_DIR = PROJECT_ROOT / "eval"

MODEL_REPO = "unsloth/gemma-3-4b-it-GGUF"
MODEL_FILE = "gemma-3-4b-it-Q4_K_M.gguf"
MODEL_DIR = MODELS_DIR / "gemma3"
MODEL_PATH = MODEL_DIR / MODEL_FILE

LLAMA_SERVER = LLAMA_DIR / "llama-server.exe"

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Model path:", MODEL_PATH)
print("llama-server:", LLAMA_SERVER)

Python: x:\dev\projects\study\ATLAS\.venv\Scripts\python.exe
Project root: X:\dev\projects\study\ATLAS
Model path: X:\dev\projects\study\ATLAS\models\gemma3\gemma-3-4b-it-Q4_K_M.gguf
llama-server: X:\dev\projects\study\ATLAS\llama\llama-server.exe


In [8]:
# import os
# import time
# import json
# import subprocess
from pathlib import Path

# import requests
# from huggingface_hub import snapshot_download
from openai import OpenAI

In [9]:
HOST = "127.0.0.1"
PORT = 8080
BASE_URL = f"http://{HOST}:{PORT}/v1"

# Для Windows + Vulkan сборки llama.cpp
SERVER_CMD = [
    str(LLAMA_SERVER),
    "-m", str(MODEL_PATH),
    "-ngl", "999",
    "-c", "8192",
    "--host", HOST,
    "--port", str(PORT),
]

LOG_PATH = EVAL_DIR / "llama_server.log"

print("Command:")
print(" ".join(SERVER_CMD))
print("Log file:", LOG_PATH)

Command:
X:\dev\projects\study\ATLAS\llama\llama-server.exe -m X:\dev\projects\study\ATLAS\models\gemma3\gemma-3-4b-it-Q4_K_M.gguf -ngl 999 -c 8192 --host 127.0.0.1 --port 8080
Log file: X:\dev\projects\study\ATLAS\eval\llama_server.log


In [13]:
import subprocess

log_f = open(LOG_PATH, "w", encoding="utf-8")
llama_process = subprocess.Popen(
    SERVER_CMD,
    cwd=str(LLAMA_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP
)
print(f"Started llama-server, PID={llama_process.pid}")

Started llama-server, PID=4704


In [14]:
import requests

r = requests.get(BASE_URL + "/models", timeout=10)
r.raise_for_status()
models_payload = r.json()
models_payload

{'models': [{'name': 'gemma-3-4b-it-Q4_K_M.gguf',
   'model': 'gemma-3-4b-it-Q4_K_M.gguf',
   'modified_at': '',
   'size': '',
   'digest': '',
   'type': 'model',
   'description': '',
   'tags': [''],
   'capabilities': ['completion'],
   'parameters': '',
   'details': {'parent_model': '',
    'format': 'gguf',
    'family': '',
    'families': [''],
    'parameter_size': '',
    'quantization_level': ''}}],
 'object': 'list',
 'data': [{'id': 'gemma-3-4b-it-Q4_K_M.gguf',
   'aliases': [],
   'tags': [],
   'object': 'model',
   'created': 1773058879,
   'owned_by': 'llamacpp',
   'meta': {'vocab_type': 1,
    'n_vocab': 262208,
    'n_ctx_train': 131072,
    'n_embd': 2560,
    'n_params': 3880263168,
    'size': 2483352832}}]}

In [15]:
client = OpenAI(
    base_url=BASE_URL,
    api_key="not-needed",
)

response = client.chat.completions.create(
    model="local-model",
    messages=[
        {
            "role": "user",
            "content": "How to write a scientific report? Answer in 2 sentences."
        }
    ],
    temperature=0.7,
    max_tokens=500,
)

print(response.choices[0].message.content)

A scientific report should clearly and concisely present your research findings, including an introduction outlining the background and purpose, a methods section detailing your procedures, results showcasing your data, and a discussion interpreting those results within the broader context of existing knowledge.  It’s crucial to maintain objectivity, use precise language, and adhere to a standardized format like IMRaD (Introduction, Methods, Results, and Discussion) to ensure your work is easily understood and verifiable by other scientists.


In [19]:
import gc
import time
import subprocess

try:
    if "llama_process" in globals() and llama_process is not None and llama_process.poll() is None:
        llama_process.terminate()
        try:
            llama_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            llama_process.kill()
            llama_process.wait(timeout=5)
except Exception:
    pass

for name in [
    "llama-server.exe",
    "llama-cli.exe",
    "llama-bench.exe",
    "llama-simple.exe",
    "llama-run.exe",
]:
    try:
        subprocess.run(["taskkill", "/F", "/T", "/IM", name], check=False, capture_output=True)
    except Exception:
        pass

for v in ["client", "response", "llm", "model", "messages", "SERVER_CMD", "llama_process"]:
    if v in globals():
        try:
            del globals()[v]
        except Exception:
            pass

gc.collect()
time.sleep(2)
subprocess.run(["nvidia-smi"], check=False)

CompletedProcess(args=['nvidia-smi'], returncode=0)